# Project 3 - Aircraft Classification

A complete development notebook that trains multiple backbones (ResNet-18 Frozen Baseline, ResNet-18 Unfrozen, ResNet-50, EfficientNet-B0, DINOv3) using a unified training and evaluation loop to compare performance on FGVC-Aircraft.

### Google Colab Setup (If applicable)

In [ ]:
# Check if the notebook is run locally or on Google Colab instance
import os
import sys

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    # Point this to your project folder in Google Drive
    BASE_DIR = '/content/drive/MyDrive/project/aircraft_classification/'
else:
    # Local or standard Jupyter server
    BASE_DIR = './'

# Prepend BASE_DIR to sys.path
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

# Dataset path
data_path = os.path.join(BASE_DIR, 'FGVCAircraft_Subset20')

In [ ]:
# Change working directory if running in Google Colab
if 'google.colab' in sys.modules and os.path.exists(BASE_DIR):
    os.chdir(BASE_DIR)
    print(f"Current working directory: {os.getcwd()}")
    print("Directory contents:", os.listdir('.'))

### Import essential libraries and settings

In [ ]:
import random
import copy
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split

from utils import train_epoch, eval_epoch, set_seed

In [ ]:
# Setup Device and Seed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
set_seed(42)

### Data Preprocessing & DataLoaders

In [ ]:
# ImageNet statistics
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

# Base transformation (Tensor conversion + Normalization)
base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

# Training data augmentation
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    base_transform
])

# Validation and Test transformation
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    base_transform
])

# Load ImageFolder datasets with corresponding transforms
trainval_raw_train = torchvision.datasets.ImageFolder(os.path.join(data_path, 'trainval'), transform=train_transform)
trainval_raw_val = torchvision.datasets.ImageFolder(os.path.join(data_path, 'trainval'), transform=val_transform)
test_dataset = torchvision.datasets.ImageFolder(os.path.join(data_path, 'test'), transform=val_transform)

# Perform stratified split (80% train, 20% validation)
indices = list(range(len(trainval_raw_train)))
labels = [label for _, label in trainval_raw_train.samples]

train_indices, val_indices = train_test_split(
    indices, test_size=0.2, stratify=labels, random_state=42
)

train_dataset = torch.utils.data.Subset(trainval_raw_train, train_indices)
val_dataset = torch.utils.data.Subset(trainval_raw_val, val_indices)

num_classes = len(trainval_raw_train.classes)
class_names = trainval_raw_train.classes

print(f"Number of classes: {num_classes}")
print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of validation samples: {len(val_dataset)}")
print(f"Number of test samples: {len(test_dataset)}")

# Create DataLoaders
batch_size = 16
num_workers = 4 if torch.cuda.is_available() and os.name != 'nt' else 2
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

### Model Factory

In [ ]:
def get_model(model_name, num_classes, freeze_backbone=False):
    """
    Initializes a model architecture and adapts the classification head for `num_classes`.
    Supported model_name: 'resnet18', 'resnet50', 'efficientnet_b0', 'dinov3'
    """
    if model_name == 'resnet18':
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        
    elif model_name == 'resnet50':
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        
    elif model_name == 'efficientnet_b0':
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        
    elif model_name == 'dinov3':
        class DinoV3Classifier(nn.Module):
            def __init__(self, repo_dir, weights, num_classes, freeze_backbone=False):
                super().__init__()
                self.backbone = torch.hub.load(
                    repo_dir,
                    'dinov3_vits16',
                    source='local',
                    weights=weights
                )
                embed_dim = getattr(self.backbone, 'embed_dim', 384)
                self.fc = nn.Linear(embed_dim, num_classes)
                
                if freeze_backbone:
                    for param in self.backbone.parameters():
                        param.requires_grad = False

            def forward(self, x):
                features = self.backbone(x)
                return self.fc(features)
        
        base = BASE_DIR if 'BASE_DIR' in globals() else './'
        repo_dir = os.path.join(base, 'dinov3') if os.path.exists(os.path.join(base, 'dinov3')) else './dinov3'
        weights_path = os.path.join(base, 'weights', 'dinov3_vits16_pretrain_lvd1689m-08c60483.pth') if os.path.exists(os.path.join(base, 'weights', 'dinov3_vits16_pretrain_lvd1689m-08c60483.pth')) else './weights/dinov3_vits16_pretrain_lvd1689m-08c60483.pth'
        
        model = DinoV3Classifier(
            repo_dir=repo_dir,
            weights=weights_path,
            num_classes=num_classes,
            freeze_backbone=freeze_backbone
        )
    else:
        raise ValueError(f"Model {model_name} not supported")
    return model

### Training and Evaluation Loop

In [ ]:
def train_and_evaluate(model, model_name, train_loader, val_loader, device, total_epochs=32, lr=0.001):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    
    # Initialize optimizer bound to trainable parameters
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs)
    
    best_acc = -float("inf")
    patience = 5
    epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    
    print(f"\n{'='*40}\nTraining: {model_name.upper()}\n{'='*40}")
    epoch_pbar = tqdm.tqdm(range(total_epochs), colour="blue")
    
    for epoch in epoch_pbar:
        # Train epoch
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, epoch, device)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        
        scheduler.step()
        
        # Validation epoch
        val_loss, val_acc = eval_epoch(model, val_loader, criterion, epoch, device)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        
        epoch_pbar.write(f"Epoch [{epoch+1:2d}/{total_epochs:2d}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        # Early Stopping & Checkpointing
        if val_acc > best_acc:
            best_acc = val_acc
            epochs_no_improve = 0
            clean_name = model_name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
            torch.save(model.state_dict(), f"weights_{clean_name}_best.pth")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered for {model_name} at epoch {epoch+1}")
                break

    # Side-by-side plots: Loss and Accuracy over Epochs
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    epochs_range = range(1, len(history["train_loss"]) + 1)

    # Loss subplot
    axes[0].plot(epochs_range, history["train_loss"], label="Train Loss", marker="o", linewidth=2)
    axes[0].plot(epochs_range, history["val_loss"], label="Validation Loss", marker="o", linewidth=2)
    axes[0].set_title(f"{model_name} - Loss over Epochs")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")

    # Accuracy subplot
    axes[1].plot(epochs_range, history["train_acc"], label="Train Accuracy", marker="x", linewidth=2)
    axes[1].plot(epochs_range, history["val_acc"], label="Validation Accuracy", marker="x", linewidth=2)
    axes[1].set_title(f"{model_name} - Accuracy over Epochs")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")

    for ax in axes:
        ax.legend()
        ax.grid(True, linestyle="--", alpha=0.7)

    plt.tight_layout()
    plt.show()

    return best_acc, history

### Model Sweep Execution & Comparison

In [ ]:
models_to_test = [
    {"name": "resnet18", "freeze": True, "label": "Baseline (ResNet-18 Frozen)"},
    {"name": "resnet18", "freeze": False, "label": "H1 (ResNet-18 Unfrozen)"},
    {"name": "resnet50", "freeze": False, "label": "H2 (ResNet-50 Unfrozen)"},
    {"name": "efficientnet_b0", "freeze": False, "label": "H3 (EfficientNet-B0 Unfrozen)"},
    {"name": "dinov3", "freeze": False, "label": "H4 (DINOv3 Unfrozen)"}
]

results = {}

for config in models_to_test:
    name = config["name"]
    freeze = config["freeze"]
    label = config["label"]
    
    model = get_model(name, num_classes=num_classes, freeze_backbone=freeze)
    
    best_acc, history = train_and_evaluate(
        model=model, 
        model_name=label,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        total_epochs=32 # Set to a lower number for quick sweeps
    )
    
    results[label] = {
        "best_val_acc": best_acc,
        "history": history
    }

print("\n" + "="*50)
print("             SWEEP COMPLETE SUMMARY")
print("="*50)
for label, res in results.items():
    print(f"{label:<35}: Best Val Acc = {res['best_val_acc']:.4f}")

with open("sweep_results.pkl", "wb") as f:
    pickle.dump(results, f)
print("\nSaved sweep results to sweep_results.pkl")